# Unit 17 / Chapter 17: Quantum Advantage and Benchmarking

> **Main Learning Objective:** Learn what "quantum advantage" actually means, understand the landmark demonstrations and their critiques, and run a mini cross-entropy benchmarking (XEB) experiment.

| Section | Topic |
|---|---|
| 17.1 | What "quantum advantage" really means |
| 17.2 | Landmark demonstrations and their critiques |
| 17.3 | How to benchmark: RB, XEB, Quantum Volume |
| 17.4 | A tiny XEB experiment |

---
## Setup

In [ ]:
# Verify libraries. Works in classic Jupyter, JupyterLite/Pyodide, and Colab.
import importlib.util
required = ["numpy", "matplotlib"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    try:
        import piplite
        await piplite.install(missing)
    except ImportError:
        try:
            import micropip
            await micropip.install(missing)
        except ImportError:
            ip = get_ipython()
            ip.run_line_magic('pip', 'install --quiet ' + ' '.join(missing))
import numpy, matplotlib
print("All libraries ready.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display, Markdown
import math, random
np.random.seed(7); random.seed(7)
plt.rcParams['figure.dpi'] = 100

# Tiny quantum simulator used across all units
def ket0(n):
    s = np.zeros(2**n, dtype=complex); s[0] = 1.0
    return s
def kron_all(mats):
    out = mats[0]
    for m in mats[1:]:
        out = np.kron(out, m)
    return out
I2 = np.eye(2, dtype=complex)
X  = np.array([[0,1],[1,0]], dtype=complex)
Y  = np.array([[0,-1j],[1j,0]], dtype=complex)
Z  = np.array([[1,0],[0,-1]], dtype=complex)
H  = (1/np.sqrt(2))*np.array([[1,1],[1,-1]], dtype=complex)
def Rx(t): c,s = np.cos(t/2), np.sin(t/2); return np.array([[c,-1j*s],[-1j*s,c]], dtype=complex)
def Ry(t): c,s = np.cos(t/2), np.sin(t/2); return np.array([[c,-s],[s,c]], dtype=complex)
def Rz(t): return np.array([[np.exp(-1j*t/2),0],[0,np.exp(1j*t/2)]], dtype=complex)
def apply_1q(gate, qubit, n):
    return kron_all([gate if i==qubit else I2 for i in range(n)])
def apply_cnot(control, target, n):
    dim = 2**n
    op = np.zeros((dim, dim), dtype=complex)
    for x in range(dim):
        bits = [(x >> (n-1-i)) & 1 for i in range(n)]
        if bits[control] == 1:
            bits[target] ^= 1
        y = 0
        for b in bits:
            y = (y<<1) | b
        op[y, x] = 1
    return op
def expZ(state, qubit, n):
    Zop = apply_1q(Z, qubit, n)
    return float(np.real(np.conj(state) @ Zop @ state))
print("Quantum simulator ready.")

---
## Course check-in

This logs that you started **Unit 17**. Enter the email you signed up with.

In [ ]:
# ============================================================
# COURSE TRACKING, do not edit
# ============================================================
import json
from urllib.request import Request, urlopen
from urllib.error  import URLError

UNIT_NUMBER = 17
TRACKER_URL = "https://script.google.com/macros/s/AKfycbyp01BDLgzqHk5HbYt7Tl0hYESKo4qRs8AMJsFKUfbNKdbUuzjT6yb1L2qVFd_oz2Ur/exec"

def _post_event(event_type, payload=None):
    body = json.dumps({
        "event_type": event_type,
        "email":      _student_email,
        "unit":       UNIT_NUMBER,
        "payload":    payload or {}
    }).encode("utf-8")
    try:
        req = Request(TRACKER_URL, data=body,
                      headers={"Content-Type": "text/plain;charset=utf-8"})
        urlopen(req, timeout=10).read()
    except URLError as e:
        print("(could not reach tracker:", e, ")")

_student_email = input("Enter the email you signed up with: ").strip().lower()
if "@" not in _student_email:
    raise ValueError("That does not look like a valid email. Re-run this cell.")

print(f"Hi {_student_email}! Logging that you started Unit {UNIT_NUMBER}.")
_post_event("unit_started")

---
# Section 17.1: What "Quantum Advantage" Really Means

"Quantum advantage" (once called "quantum supremacy," a term now discouraged) means a task where a quantum computer beats the best possible classical computer. It sounds simple. It is not. Several kinds of advantage are worth distinguishing:

* **Asymptotic advantage**: for large enough problem sizes, the quantum algorithm scales better. Shor's algorithm for factoring is exponentially better asymptotically. But at today's small sizes, classical wins.
* **Practical advantage**: on inputs that fit on today's hardware, the quantum computer finishes faster or with less energy. This is a much stricter bar.
* **Sample-complexity advantage**: for a given accuracy, the quantum algorithm needs fewer training samples. This is the QML version.
* **Energy advantage**: uses less watt-hours. Google's Sycamore result was framed here.

Whenever someone claims quantum advantage, ask: (a) which of these do they mean, (b) what is the best classical baseline they compared against, and (c) is the task useful or a synthetic benchmark chosen to favor quantum?

---
# Section 17.2: Landmark Demonstrations and Their Critiques

Three headline experiments:

**Google Sycamore (2019).** A 53-qubit chip sampled from a random-circuit distribution in 200 seconds. Google claimed the same task would take a classical supercomputer 10000 years. IBM immediately responded with a better classical algorithm that would take days, not millennia. So the claim was a moving target; both sides made progress.

**USTC Jiuzhang (2020 and 2021).** A photonic system did **Gaussian boson sampling** faster than any classical machine. This was an asymptotic advantage but on a specific, contrived sampling task with no known application.

**Google Willow (2024).** A 105-qubit chip demonstrated **exponential error suppression** as the surface code distance grew. This was a milestone toward *fault tolerance*, not a raw speed claim.

The pattern: every advantage claim gets scrutinized, sometimes matched by better classical algorithms, and refined. That is healthy science but it means "quantum advantage" is always provisional. The bar keeps rising as classical algorithms improve.

---
# Section 17.3: How to Benchmark: RB, XEB, Quantum Volume

You cannot make claims about a quantum computer without benchmarks. Three widely-used ones:

**Randomized Benchmarking (RB).** Apply a random sequence of Clifford gates that should compose to the identity. If the output stays near |0>, gates are accurate. If it drifts, extract an average per-gate error. Simple, hardware-agnostic, standard for reporting single-qubit and two-qubit gate errors.

**Cross-Entropy Benchmarking (XEB).** Google's flagship benchmark. Run a random circuit, sample its outputs, and compare the empirical bitstring distribution to the ideal one. Report an "XEB fidelity" that measures how much the real distribution overlaps the ideal.

**Quantum Volume (QV).** IBM's flagship. QV is the largest N such that a random N-qubit circuit of depth N runs with fidelity > 2/3. It captures gates, connectivity, and noise all in one number. Every year IBM's chips announce a bigger QV; today's flagship machines are around QV 128.

Below we implement a mini XEB experiment on a simulated noisy circuit.

In [ ]:
def random_circuit(n_qubits, depth, seed=0):
    rng = np.random.default_rng(seed)
    state = ket0(n_qubits)
    for layer in range(depth):
        for q in range(n_qubits):
            # random single-qubit rotation
            a, b, c = rng.uniform(0, 2*np.pi, size=3)
            gate = Rz(c) @ Ry(b) @ Rz(a)
            state = apply_1q(gate, q, n_qubits) @ state
        # entangling layer: CNOT ladder
        for q in range(n_qubits - 1):
            state = apply_cnot(q, q+1, n_qubits) @ state
    return state

def apply_noise(state, p_error):
    # Depolarizing-ish: with prob p, flip a random qubit
    if np.random.random() < p_error:
        n = int(np.log2(len(state)))
        q = np.random.randint(n)
        state = apply_1q(X, q, n) @ state
    return state

def xeb_fidelity(ideal, samples, n):
    # samples: list of bitstring indices, one per shot (drawn from noisy device)
    probs_ideal = np.abs(ideal)**2
    dim = 2**n
    return dim * np.mean(probs_ideal[samples]) - 1

# One noisy run
n = 4; depth = 6
ideal_state = random_circuit(n, depth, seed=7)
ideal_probs = np.abs(ideal_state)**2

# Simulate the noisy device: with noise, sample from a mixed distribution
noisy_probs = 0.7 * ideal_probs + 0.3 / (2**n)  # 30% depolarizing
noisy_probs /= noisy_probs.sum()
n_shots = 5000
samples = np.random.choice(2**n, size=n_shots, p=noisy_probs)

fid = xeb_fidelity(ideal_state, samples, n)
print(f"XEB fidelity (0 = pure noise, 1 = perfect): {fid:.3f}")

### Activity 17.1

Run the same circuit at three noise levels (10 percent, 30 percent, 50 percent depolarizing) and print the XEB fidelity for each. What do you expect the trend to be?

In [ ]:
# TODO: loop over three noise levels and report XEB fidelity.
noise_levels = [0.1, 0.3, 0.5]
results = None

if results is None:
    results = []
    for p in noise_levels:
        pn = (1 - p) * ideal_probs + p / (2**n)
        pn /= pn.sum()
        s  = np.random.choice(2**n, size=5000, p=pn)
        results.append(xeb_fidelity(ideal_state, s, n))
    for p, r in zip(noise_levels, results):
        print(f"noise {p:.1f} -> XEB fidelity {r:+.3f}")

<details><summary>Solution</summary>

Higher noise means the noisy distribution looks more like the uniform distribution, so XEB fidelity drops toward 0. You should see approximately 0.9, 0.7, 0.5 for the three noise levels.
</details>

---
# Section 17.4: When Is an XEB Result Really "Advantage"?

Just running the circuit and computing XEB is not enough. To claim advantage you need:

1. A **classical baseline** for how long the best known classical simulator would take.
2. Enough shots that your XEB estimate has small statistical error.
3. Proof the circuit is hard to simulate classically (usually shown by circuit-depth or entanglement arguments).
4. Reproducibility on independent runs.

Google's Sycamore XEB was 0.002 with millions of shots. IBM's counter-simulation showed classical could still match it in a few days, so the "advantage margin" was smaller than the initial 10000-year headline.

### Activity 17.2

In one sentence, explain why an XEB fidelity of 0.002 can still constitute a quantum advantage claim, even though 0.002 sounds tiny.

In [ ]:
answer_17_2 = """YOUR ANSWER HERE, one or two sentences."""
print(answer_17_2)

<details><summary>Sample answer</summary>

XEB fidelity is normalized so that pure noise scores 0. A fidelity of 0.002 means the empirical distribution genuinely reflects the ideal circuit rather than pure noise, and doing this at 53-qubit scale is the hard part. What matters is not the magnitude but whether it beats what a classical machine can do in the same time.
</details>

### Activity 17.3

What benchmarks would you want to see reported alongside a claim like "our new QML algorithm outperforms classical on task X"? Name three.

In [ ]:
answer_17_3 = """YOUR ANSWER HERE, list three benchmarks."""
print(answer_17_3)

<details><summary>Sample answer</summary>

(1) The exact classical baseline algorithm and its hyperparameters, (2) sample complexity for both methods (how many training examples were used), and (3) hardware resource usage (qubit count, gate count, wall-clock time). Bonus: whether the task admits a dequantization argument.
</details>

---
## Section summary

* "Quantum advantage" has multiple meanings; always ask which and against what baseline.
* Google Sycamore, USTC Jiuzhang, and Google Willow are landmark demonstrations, each with limits.
* Randomized Benchmarking, XEB, and Quantum Volume are the standard tools for measuring quantum hardware.
* Meaningful advantage claims require classical baselines, statistical rigor, and reproducibility.

---
## End-of-Unit Quiz (10 multiple choice)

**Q1.** "Quantum advantage" preferred over "quantum supremacy" because:

A. It is a shorter word
B. It is more accurate and less politically loaded
C. It is required by the IEEE
D. It sounds cooler

**Q2.** Which of these is NOT a type of quantum advantage?

A. Asymptotic
B. Practical
C. Aesthetic
D. Sample complexity

**Q3.** Google's 2019 Sycamore result was contested because:

A. IBM found a better classical algorithm reducing the classical time from 10000 years to days
B. The chip did not exist
C. It used only 3 qubits
D. Photons cannot factor

**Q4.** USTC Jiuzhang demonstrated advantage on:

A. RSA factoring
B. Gaussian boson sampling
C. Image classification
D. Weather forecasting

**Q5.** Randomized Benchmarking measures:

A. The maximum entanglement
B. Average per-gate error via random Clifford sequences
C. The number of qubits
D. The temperature

**Q6.** Quantum Volume captures:

A. The largest random circuit that runs with fidelity above 2/3
B. Only the qubit count
C. Only the gate count
D. Only the connectivity

**Q7.** XEB fidelity of 0 means:

A. Perfect circuit
B. The distribution is pure noise
C. Best possible score
D. The circuit crashed

**Q8.** In our toy experiment, higher depolarizing noise made XEB fidelity:

A. Higher
B. Lower, toward 0
C. Unchanged
D. Negative

**Q9.** Google Willow's 2024 milestone was primarily about:

A. Speed of sampling
B. Exponential logical-error suppression with growing surface code
C. First 1000-qubit chip
D. Beating Shor's algorithm

**Q10.** To evaluate a QML "advantage" claim, you should ask about:

A. Only the quantum runtime
B. Only the qubit count
C. The classical baseline, sample complexity, and hardware resource use
D. Only the paper's citation count

---
## End-of-unit submission

Fill in your ten multiple choice answers, then run this cell to submit.

In [ ]:
quiz_answers = {
    "q1":  "",   # A, B, C, or D
    "q2":  "",
    "q3":  "",
    "q4":  "",
    "q5":  "",
    "q6":  "",
    "q7":  "",
    "q8":  "",
    "q9":  "",
    "q10": ""
}

reflection = "What did you find most interesting in this unit? (optional)"

_post_event("unit_completed",
            payload={"quiz": quiz_answers, "reflection": reflection})

print(f"Submitted Unit 17!")